# 02 — Metrics Deep Dive

This notebook covers:

1. Listing available metrics
2. Building `EvalCase` inputs for RAG evaluation
3. Running all built-in metrics with a deterministic `ReplayJudge`
4. Inspecting per-metric scores, reasons, and trial details
5. Understanding skip behavior for missing inputs
6. Running the **C4** composite metric

No API key is needed — all calls use a `ReplayJudge` that returns
pre-defined structured outputs.

In [1]:
from pprint import pprint

from evalf import EvalCase, Evaluator
from evalf.llms.base import BaseLLMModel
from evalf.metrics import (
    AnswerCorrectnessMetric,
    AnswerRelevanceMetric,
    C4Metric,
    ContextCoverageMetric,
    ContextPrecisionMetric,
    ContextRecallMetric,
    ContextRelevanceMetric,
    FaithfulnessMetric,
    list_metric_names,
)
from evalf.metrics.answer_correctness.schema import AnswerCorrectnessAssessment
from evalf.metrics.answer_relevance.schema import AnswerRelevanceAssessment
from evalf.metrics.c4.schema import C4Assessment, CriterionAssessment
from evalf.metrics.context_coverage.schema import (
    ContextCoverageAssessment as CoverageMetricAssessment,
)
from evalf.metrics.decomposition import (
    Claim,
    ClaimCoverageAssessment,
    ClaimCoverageVerdict,
    ClaimExtraction,
    ClaimSupportAssessment,
    ClaimSupportVerdict,
    ContextCoverageAssessment,
    ContextCoverageVerdict,
    ContextRelevanceVerdict,
    ContextRelevanceVerdictList,
)
from evalf.schemas import LLMResponse, UsageStats

## 1. Available metrics

In [2]:
list_metric_names()

['answer_correctness',
 'answer_relevance',
 'c4',
 'context_coverage',
 'context_precision',
 'context_recall',
 'context_relevance',
 'faithfulness']

## 2. ReplayJudge for deterministic testing

The `ReplayJudge` pops pre-built outputs in FIFO order,
so the notebook produces identical results every time.

In [3]:
class ReplayJudge(BaseLLMModel):
    """Deterministic judge that replays a fixed sequence of structured outputs."""

    def __init__(self, outputs):
        super().__init__(
            provider="test",
            model="replay-judge",
            base_url="https://example.com",
            api_key=None,
        )
        self._outputs = list(outputs)

    def _pop_response(self):
        if not self._outputs:
            raise AssertionError("No replay outputs left.")
        return LLMResponse(
            text=None,
            model=self.model,
            provider=self.provider,
            parsed_output=self._outputs.pop(0),
            usage=UsageStats(
                input_tokens=20,
                output_tokens=10,
                total_tokens=30,
                latency_ms=12.5,
            ),
        )

    def generate(self, **kwargs):
        return self._pop_response()

    async def a_generate(self, **kwargs):
        return self._pop_response()

## 3. Build a test case and metrics

In [4]:
case = EvalCase(
    id="case-ferpa-1",
    question="Under FERPA, when do rights transfer from parents to a student?",
    retrieved_contexts=[
        "FERPA rights transfer when a student turns 18 years old.",
        "FERPA is a federal law that protects education records.",
        "FERPA rights also transfer when the student enters a postsecondary institution at any age.",
    ],
    reference_contexts=[
        "FERPA rights transfer when a student turns 18 years old.",
        "FERPA rights also transfer when the student enters a postsecondary institution at any age.",
    ],
    actual_output=(
        "FERPA rights transfer when a student turns 18 or when they enter "
        "a postsecondary institution at any age."
    ),
    expected_output=(
        "FERPA rights transfer when a student turns 18 years old or enters "
        "a postsecondary institution at any age."
    ),
)

metrics = [
    FaithfulnessMetric(threshold=0.8),
    AnswerCorrectnessMetric(threshold=0.8),
    AnswerRelevanceMetric(threshold=0.8),
    ContextCoverageMetric(threshold=0.7),
    ContextRelevanceMetric(threshold=0.7),
    ContextPrecisionMetric(threshold=0.8),
    ContextRecallMetric(threshold=0.8),
]
print(f"{len(metrics)} metrics configured")

7 metrics configured


## 4. Replay judge outputs

Each metric consumes one or more structured outputs from the queue.
Single-call metrics (answer_correctness, answer_relevance, context_coverage, context_relevance)
use one judge call. Decomposed metrics (faithfulness, context_precision, context_recall)
use two judge calls each: one for claim extraction, one for verification.

In [5]:
judge_outputs = [
    # --- faithfulness: extract claims, then verify ---
    ClaimExtraction(
        claims=[
            Claim(claim_id="c1", text="FERPA rights transfer when a student turns 18 years old."),
            Claim(
                claim_id="c2",
                text="FERPA rights also transfer when the student enters a postsecondary institution at any age.",
            ),
        ]
    ),
    ClaimSupportAssessment(
        verdicts=[
            ClaimSupportVerdict(
                claim_id="c1",
                verdict="supported",
                evidence_context_ids=["ctx_1"],
                reason="Supported by first context.",
            ),
            ClaimSupportVerdict(
                claim_id="c2",
                verdict="supported",
                evidence_context_ids=["ctx_3"],
                reason="Supported by third context.",
            ),
        ]
    ),
    # --- answer_correctness ---
    AnswerCorrectnessAssessment(score=1.0, reason="Both FERPA conditions are present."),
    # --- answer_relevance ---
    AnswerRelevanceAssessment(score=1.0, reason="Directly answers the question."),
    # --- context_coverage (single call) ---
    CoverageMetricAssessment(
        score=1.0,
        verdict="yes",
        reason="Both key FERPA transfer conditions (age 18 and postsecondary) are present in the retrieval contexts.",
    ),
    # --- context_relevance ---
    ContextRelevanceVerdictList(
        verdicts=[
            ContextRelevanceVerdict(
                context_id="ctx_1", verdict="relevant", reason="Age-18 condition."
            ),
            ContextRelevanceVerdict(
                context_id="ctx_2", verdict="partially_relevant", reason="FERPA background."
            ),
            ContextRelevanceVerdict(
                context_id="ctx_3", verdict="relevant", reason="Postsecondary condition."
            ),
        ]
    ),
    # --- context_precision: extract ref claims, then assess contexts ---
    ClaimExtraction(
        claims=[
            Claim(claim_id="rc1", text="FERPA rights transfer when a student turns 18 years old."),
            Claim(
                claim_id="rc2",
                text="FERPA rights also transfer when the student enters a postsecondary institution at any age.",
            ),
        ]
    ),
    ContextCoverageAssessment(
        contexts=[
            ContextCoverageVerdict(
                context_id="ctx_1", supported_claim_ids=["rc1"], reason="Covers age-18."
            ),
            ContextCoverageVerdict(
                context_id="ctx_2", supported_claim_ids=[], reason="General background."
            ),
            ContextCoverageVerdict(
                context_id="ctx_3", supported_claim_ids=["rc2"], reason="Covers postsecondary."
            ),
        ]
    ),
    # --- context_recall: extract ref claims, then verify coverage ---
    ClaimExtraction(
        claims=[
            Claim(claim_id="rc1", text="FERPA rights transfer when a student turns 18 years old."),
            Claim(
                claim_id="rc2",
                text="FERPA rights also transfer when the student enters a postsecondary institution at any age.",
            ),
        ]
    ),
    ClaimCoverageAssessment(
        verdicts=[
            ClaimCoverageVerdict(
                claim_id="rc1",
                verdict="supported",
                evidence_context_ids=["ctx_1"],
                reason="Covered.",
            ),
            ClaimCoverageVerdict(
                claim_id="rc2",
                verdict="supported",
                evidence_context_ids=["ctx_3"],
                reason="Covered.",
            ),
        ]
    ),
]

## 5. Run the evaluator and inspect results

In [6]:
report = await Evaluator(judge=ReplayJudge(judge_outputs), concurrency=1).a_evaluate(
    cases=[case],
    metrics=metrics,
)

print("=== Run Summary ===")
pprint(report.summary.model_dump(exclude_none=True))

=== Run Summary ===
{'avg_latency_ms_per_sample': 125.0,
 'failed_samples': 0,
 'metric_pass_rates': {'answer_correctness': 1.0,
                       'answer_relevance': 1.0,
                       'context_coverage': 1.0,
                       'context_precision': 1.0,
                       'context_recall': 1.0,
                       'context_relevance': 1.0,
                       'faithfulness': 1.0},
 'passed_samples': 1,
 'skipped_samples': 0,
 'total_input_tokens': 200,
 'total_output_tokens': 100,
 'total_samples': 1,
 'total_tokens': 300}


In [7]:
print("=== Per-metric results ===")
for m in report.samples[0].metrics:
    print(f"  {m.name:25s}  status={m.status:6s}  score={m.score}  reason={m.reason}")

=== Per-metric results ===
  faithfulness               status=passed  score=1.0  reason=Passed in 1/1 evaluated attempt(s) under pass@k.
  answer_correctness         status=passed  score=1.0  reason=Passed in 1/1 evaluated attempt(s) under pass@k.
  answer_relevance           status=passed  score=1.0  reason=Passed in 1/1 evaluated attempt(s) under pass@k.
  context_coverage           status=passed  score=1.0  reason=Passed in 1/1 evaluated attempt(s) under pass@k.
  context_relevance          status=passed  score=0.8333333333333334  reason=Passed in 1/1 evaluated attempt(s) under pass@k.
  context_precision          status=passed  score=0.8333333333333333  reason=Passed in 1/1 evaluated attempt(s) under pass@k.
  context_recall             status=passed  score=1.0  reason=Passed in 1/1 evaluated attempt(s) under pass@k.


In [8]:
print("=== Detailed trial results (faithfulness) ===")
faith = report.samples[0].metrics[0]
pprint(faith.model_dump(exclude_none=True))

=== Detailed trial results (faithfulness) ===
{'best_trial_score': 1.0,
 'evaluated_k': 1,
 'mean_trial_score': 1.0,
 'missing_inputs': [],
 'mode': 'pass@k',
 'name': 'faithfulness',
 'passed': True,
 'reason': 'Passed in 1/1 evaluated attempt(s) under pass@k.',
 'requested_k': 1,
 'required_inputs': ['question', 'retrieved_contexts', 'actual_output'],
 'score': 1.0,
 'status': 'passed',
 'successful_trials': 1,
 'threshold': 0.8,
 'trial_results': [{'attempt_index': 1,
                    'missing_inputs': [],
                    'passed': True,
                    'reason': 'Supported 2/2 material claim(s).',
                    'score': 1.0,
                    'status': 'passed',
                    'usage': {'input_tokens': 40,
                              'latency_ms': 25.0,
                              'output_tokens': 20,
                              'total_tokens': 60}}],
 'usage': {'input_tokens': 40,
           'latency_ms': 25.0,
           'output_tokens': 20,
        

## 6. Skip behavior

Metrics declare `required_inputs`. If a field is missing or blank,
the metric is **skipped** before any judge call.

In [9]:
skipped_report = await Evaluator(judge=ReplayJudge([])).a_evaluate(
    cases=[
        EvalCase(
            id="case-skip",
            question="When do FERPA rights transfer?",
            actual_output="FERPA rights transfer at age 18.",
        )
    ],
    metrics=[ContextRecallMetric()],
)

skipped = skipped_report.samples[0].metrics[0]
print(f"Status: {skipped.status}")
print(f"Missing inputs: {skipped.missing_inputs}")
print(f"Required inputs: {skipped.required_inputs}")

Status: skipped
Missing inputs: ['reference_contexts', 'retrieved_contexts']
Required inputs: ['question', 'retrieved_contexts', 'reference_contexts']


## 7. C4 composite metric

C4 scores four rubric dimensions in one judge call:
`alignment_integrity`, `accuracy_consistency`,
`safety_sovereignty_tone`, `completeness_coverage`.

Options:
- `include_reason=True` — include per-criterion reasoning in the final reason
- `need_summary_reason=True` — request a second judge call to synthesize a single summary
- `strict_mode=True` — clamp below-threshold scores to 0.0

In [10]:
c4_case = EvalCase(
    id="case-c4",
    question="Does the VinFast VF 8 support OTA updates?",
    actual_output=(
        "The VinFast VF 8 supports over-the-air software updates. "
        "You can check the version in the VinFast app."
    ),
    expected_output=(
        "The VinFast VF 8 supports over-the-air software updates. "
        "Users can verify the current version in the system settings "
        "or the official VinFast app."
    ),
)

c4_output = C4Assessment(
    alignment_integrity=CriterionAssessment(score=1.0, reasoning="On-topic, correct entity."),
    accuracy_consistency=CriterionAssessment(
        score=0.75, reasoning="Mostly correct but omits system settings."
    ),
    safety_sovereignty_tone=CriterionAssessment(score=1.0, reasoning="Safe and professional."),
    completeness_coverage=CriterionAssessment(
        score=0.75, reasoning="Misses system settings as alternative."
    ),
)

c4_report = await Evaluator(judge=ReplayJudge([c4_output]), concurrency=1).a_evaluate(
    cases=[c4_case],
    metrics=[C4Metric(threshold=0.7, include_reason=True)],
)

c4_result = c4_report.samples[0].metrics[0]
print(f"Score: {c4_result.score}")
print(f"Status: {c4_result.status}")
print(f"Reason: {c4_result.reason}")

Score: 0.875
Status: passed
Reason: alignment_integrity=1.00: On-topic, correct entity. | accuracy_consistency=0.75: Mostly correct but omits system settings. | safety_sovereignty_tone=1.00: Safe and professional. | completeness_coverage=0.75: Misses system settings as alternative.
